# 05 - Model Training (Late Fusion Architecture)

This notebook demonstrates Late Fusion model training:

**NEW ARCHITECTURE:** Two parallel models instead of single fused model
1. **Genomic Model:** Trained on MI+PSO selected genes ONLY
2. **Clinical Model:** Trained on engineered clinical features ONLY
3. **Fusion Layer:** Combines probabilities using weighted average

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import joblib
import config
from src.io import save_table, logger
from src.models import make_xgb, tune_model
from src.fusion.late_fusion import LateFusionPredictor, evaluate_late_fusion
from src.visualization import setup_style
setup_style()

## Step 1: Load Feature-Selected Data

In [ ]:
X_train_genomic = pd.read_csv(config.PROCESSED_DIR / "X_train_genomic_selected.csv")
X_test_genomic = pd.read_csv(config.PROCESSED_DIR / "X_test_genomic_selected.csv")
X_train_clinical = pd.read_csv(config.PROCESSED_DIR / "X_train_clinical_engineered.csv")
X_test_clinical = pd.read_csv(config.PROCESSED_DIR / "X_test_clinical_engineered.csv")
y_train = pd.read_csv(config.PROCESSED_DIR / "y_train.csv").iloc[:, 0]
y_test = pd.read_csv(config.PROCESSED_DIR / "y_test.csv").iloc[:, 0]

print(f"Genomic - Train: {X_train_genomic.shape}, Test: {X_test_genomic.shape}")
print(f"Clinical - Train: {X_train_clinical.shape}, Test: {X_test_clinical.shape}")
print(f"Positives - Train: {int(y_train.sum())}, Test: {int(y_test.sum())}")

## Step 2: Train Genomic Branch Model

In [ ]:
genomic_model, genomic_search = tune_model(
    "XGBoost",
    X_train_genomic,
    y_train,
    config.XGBOOST_SEARCH_SPACE,
)

print(f"Genomic Model - Best CV AUC: {genomic_search.best_score_:.4f}")
print(f"Genomic Model - Best params: {genomic_search.best_params_}")

## Step 3: Train Clinical Branch Model

In [ ]:
clinical_model, clinical_search = tune_model(
    "XGBoost",
    X_train_clinical,
    y_train,
    config.XGBOOST_SEARCH_SPACE,
)

print(f"Clinical Model - Best CV AUC: {clinical_search.best_score_:.4f}")
print(f"Clinical Model - Best params: {clinical_search.best_params_}")

## Step 4: Create Late Fusion Predictor and Optimize Weights

In [ ]:
fusion_predictor = LateFusionPredictor(
    genomic_model=genomic_model,
    clinical_model=clinical_model,
    genomic_weight=0.5,
    clinical_weight=0.5,
)

opt_gen_w, opt_clin_w, val_auc = fusion_predictor.optimize_weights(
    X_train_genomic, X_train_clinical, y_train,
    n_steps=100,
)

print(f"\nOptimized weights: genomic={opt_gen_w:.3f}, clinical={opt_clin_w:.3f}")
print(f"Validation AUC with optimized weights: {val_auc:.4f}")

## Step 5: Evaluate on Test Set

In [ ]:
results = evaluate_late_fusion(
    fusion_predictor,
    X_test_genomic,
    X_test_clinical,
    y_test,
)

print("\n=== Late Fusion Test Results ===")
print(f"Fusion AUC:      {results['fusion_auc']:.4f}")
print(f"Fusion Accuracy: {results['fusion_accuracy']:.4f}")
print(f"Genomic AUC:     {results['genomic_auc']:.4f}")
print(f"Clinical AUC:    {results['clinical_auc']:.4f}")
print(f"Weights:         genomic={results['genomic_weight']:.3f}, clinical={results['clinical_weight']:.3f}")

## Step 6: Save Models and Artifacts

In [ ]:
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
config.TABLES_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(genomic_model, config.MODELS_DIR / "genomic_model.joblib")
joblib.dump(clinical_model, config.MODELS_DIR / "clinical_model.joblib")
joblib.dump(fusion_predictor, config.MODELS_DIR / "late_fusion_predictor.joblib")

pd.DataFrame({"feature": X_train_genomic.columns.tolist()}).to_csv(
    config.TABLES_DIR / "final_genomic_features.csv", index=False
)
pd.DataFrame({"feature": X_train_clinical.columns.tolist()}).to_csv(
    config.TABLES_DIR / "final_clinical_features.csv", index=False
)

pd.DataFrame([{
    "genomic_weight": results["genomic_weight"],
    "clinical_weight": results["clinical_weight"],
    "fusion_auc": results["fusion_auc"],
}]).to_csv(config.TABLES_DIR / "fusion_weights.csv", index=False)

print("Saved:")
print(f"  - genomic_model.joblib")
print(f"  - clinical_model.joblib")
print(f"  - late_fusion_predictor.joblib")
print(f"  - final_genomic_features.csv")
print(f"  - final_clinical_features.csv")
print(f"  - fusion_weights.csv")